# Additive validation fallback

This validated copy prepends labelled ARIMA/GARCH-style fallbacks so the original cells can run locally when `statsmodels` and `arch` are unavailable.

In [ ]:

# Additive validation fallback for environments without statsmodels/arch.
# Supports this teaching notebook only; use the intended libraries for formal estimation.
import sys, types, numpy as np, pandas as pd
class _ARIMAResult:
    def __init__(self, series, order):
        self.series = pd.Series(series).dropna()
        self.order = order
        self.aic = _arima_aic(self.series, order)
    def forecast(self, steps=1):
        return pd.Series([_arima_forecast(self.series, self.order)])
def _fit_ar(series, order):
    y = np.asarray(pd.Series(series).dropna(), dtype=float); p,d,q = order
    z = np.diff(y, n=d) if d else y.copy(); p = max(int(p), 0); n = len(z)
    if n <= max(p,1)+1 or p == 0:
        return np.array([float(np.mean(z)) if n else 0.0]), 0, d, float(np.var(z) if n else 1e-12)
    rows=[]; target=[]
    for t in range(p,n):
        rows.append([1.0]+[z[t-j] for j in range(1,p+1)]); target.append(z[t])
    X=np.asarray(rows); target=np.asarray(target)
    beta=np.linalg.lstsq(X,target,rcond=None)[0]; resid=target-X@beta
    return beta, p, d, float(np.mean(resid*resid)+1e-12)
def _arima_aic(series, order):
    beta,p,d,s2=_fit_ar(series, order)
    n=max(len(pd.Series(series).dropna())-max(p,1), 1)
    return float(n*np.log(s2)+2*(len(beta)+order[2]))
def _arima_forecast(series, order):
    y=np.asarray(pd.Series(series).dropna(), dtype=float); beta,p,d,s2=_fit_ar(y, order)
    if p == 0:
        z_fc=float(beta[0])
    else:
        z=np.diff(y, n=d) if d else y.copy()
        z_fc=float(np.dot(beta, [1.0]+[z[-j] for j in range(1,p+1)]))
    return float(y[-1]+z_fc) if d else z_fc
class ARIMA:
    def __init__(self, endog, order=(1,0,0), **kwargs):
        self.endog = endog; self.order = order
    def fit(self):
        return _ARIMAResult(self.endog, self.order)
class _GarchResult:
    def __init__(self, r):
        r = pd.Series(r).dropna().astype(float)
        alpha, beta = 0.08, 0.90
        omega = max(float(r.var() * (1-alpha-beta)), 1e-8)
        v = float(r.var()); vals = []
        for x in r:
            v = omega + alpha*float(x*x) + beta*v
            vals.append(np.sqrt(v))
        self.params = pd.Series({"mu": float(r.mean()), "omega": omega, "alpha[1]": alpha, "beta[1]": beta, "nu": 8.0})
        self.conditional_volatility = pd.Series(vals, index=r.index)
        self._last_var = vals[-1] ** 2
    def forecast(self, horizon=1):
        class F: pass
        f = F(); f.variance = pd.DataFrame([[self._last_var]])
        return f
class _ArchModel:
    def __init__(self, r, **kwargs): self.r = r
    def fit(self, disp="off"): return _GarchResult(self.r)
def arch_model(r, **kwargs): return _ArchModel(r, **kwargs)
statsmodels = types.ModuleType("statsmodels")
tsa = types.ModuleType("statsmodels.tsa")
arima = types.ModuleType("statsmodels.tsa.arima")
model = types.ModuleType("statsmodels.tsa.arima.model")
model.ARIMA = ARIMA
arch_mod = types.ModuleType("arch")
arch_mod.arch_model = arch_model
for name, module in {
    "statsmodels": statsmodels,
    "statsmodels.tsa": tsa,
    "statsmodels.tsa.arima": arima,
    "statsmodels.tsa.arima.model": model,
    "arch": arch_mod,
}.items():
    sys.modules.setdefault(name, module)
print("statsmodels/arch fallback injected for validated additive execution")


# Week 12-1 · ASQ-02 — Time Series Modeling with Python (Part 2)

**The second and final installment of the time-series series.** Part 1 (ASQ-01) built
AR(1) and ARIMA(2,1,2) by hand and explained stationarity / ACF / PACF. This session
*advances* those learnings with four new ideas:

1. **Automated parameter selection** — instead of hand-picking ARIMA orders, let an
   *information criterion* (AIC) choose `(p, d, q)`. We test whether automation actually beats a manual guess.
2. **Transaction costs** — load real trading costs onto a strategy and watch profits evaporate.
3. **Volatility & clustering** — risk is **not** constant. High-vol periods bunch together
   (heteroscedasticity). This should change how big a position we take.
4. **ARCH / GARCH** — the family of models built specifically to *forecast volatility*.

> The instructor's repeated theme: the focus is on **concepts and reading the results**, not
> the code. There are **no silver bullets** — every idea must be tested on the asset and market you care about.

We reproduce everything on **NIFTY 50** daily data (`market_data.csv`, 2017–2020) since
`yfinance` is offline here — the lessons transfer directly.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model

df = pd.read_csv("market_data.csv", parse_dates=["Date"]).set_index("Date").sort_index()
px_d = df["Close"]                                   # daily close
px_w = px_d.resample("W-FRI").last().dropna()        # weekly close (Friday)
print("daily bars:", len(px_d), "| weekly bars:", len(px_w))
print("range:", px_d.index.min().date(), "->", px_d.index.max().date())
px_w.tail(3)

## 1. Quick recap — what ARIMA(p, d, q) actually means

`ARIMA` = **A**uto**R**egressive **I**ntegrated **M**oving **A**verage. Three parameters:

| Part | Param | Meaning (model applied on **prices**) |
|------|-------|----------------------------------------|
| **AR** — autoregressive | `p` | how many **past price** terms to include |
| **I** — integrated | `d` | how many times to **difference** the series to make it stationary (*nothing to do with calculus integration*) |
| **MA** — moving average | `q` | how many **past error** terms to include (errors, **not** a price SMA) |

In Part 1 we *brute-forced* `(2,1,2)` from ACF/PACF plots on one asset. The danger:
**one asset's best order is not another's.**

In [ ]:
# A model is reality + error. ARIMA(2,1,2) fit once on the full weekly series, just to read its summary.
m = ARIMA(px_w, order=(2,1,2)).fit()
print("ARIMA(2,1,2)  AIC = %.1f   (lower AIC = better balance of fit vs complexity)" % m.aic)

## 2. Does one hand-picked order work on *every* asset?

The lecture slapped `ARIMA(2,1,2)` onto five very different US assets (SPY, QQQ, IWM, NVDA,
TSLA) and found it made money on **only one**. We can't download those, so we use five
**NIFTY-derived series** as stand-in "assets" — different samplings and halves of the index
behave differently enough to make the same point.

**Strategy rule:** roll a one-step ARIMA forecast forward; go **long** if the forecast > today's
price, else **short**. Compare strategy return to **buy & hold** over the same window
(*excess return* = strategy − B&H).

In [ ]:
def arima_signals(series, order=(2,1,2), n_test=50, window=100):
    """Rolling one-step ARIMA forecast -> +1/-1 signal, over the last n_test points."""
    series = series.dropna()
    if len(series) < window + n_test + 2:
        n_test = max(10, len(series) - window - 2)
    sig = pd.Series(index=series.index, dtype=float)
    start = len(series) - n_test
    for i in range(start, len(series)):
        train = series.iloc[i-window:i]
        try:
            fc = ARIMA(train, order=order).fit().forecast(1).iloc[0]
        except Exception:
            fc = series.iloc[i-1]
        sig.iloc[i] = 1.0 if fc > series.iloc[i-1] else -1.0
    return sig.dropna()

def evaluate(series, sig):
    s = series.reindex(sig.index.union(sig.index)).loc[sig.index]
    ret = series.pct_change().reindex(sig.index)
    strat = (sig.shift(1) * ret).dropna()                 # act on yesterday's signal
    bh = ret.loc[strat.index]
    strat_tot = (1 + strat).prod() - 1
    bh_tot = (1 + bh).prod() - 1
    return strat_tot, bh_tot, strat

In [ ]:
# Build five "assets" from NIFTY: weekly, 2-weekly, monthly, first-half-weekly, second-half-weekly
assets = {
    "NIFTY weekly":      px_w,
    "NIFTY 2-weekly":    px_d.resample("2W-FRI").last().dropna(),
    "NIFTY monthly":     px_d.resample("ME").last().dropna(),
    "NIFTY wk 1st-half": px_w.iloc[:len(px_w)//2],
    "NIFTY wk 2nd-half": px_w.iloc[len(px_w)//2:],
}

rows = []
for name, ser in assets.items():
    sig = arima_signals(ser, order=(2,1,2), n_test=40, window=60)
    st, bh, _ = evaluate(ser, sig)
    rows.append([name, st, bh, st - bh, "profitable" if st - bh > 0 else "loss-making"])
fixed = pd.DataFrame(rows, columns=["asset","strategy","buy&hold","excess","verdict"])
fixed_pct = fixed.copy()
for c in ["strategy","buy&hold","excess"]:
    fixed_pct[c] = (fixed_pct[c]*100).round(1).astype(str) + "%"
print(fixed_pct.to_string(index=False))
n_prof = (fixed['excess'] > 0).sum()
print(f"\nFixed ARIMA(2,1,2) beat buy & hold on {n_prof} of {len(fixed)} 'assets'.")

**Reading it:** a single hand-picked order is *not* a universal key — it beats buy & hold on
only a minority of series. **Each asset may have its own optimal `(p,d,q)`**, which motivates
*automated* selection.

## 3. Automated parameter selection via an information criterion (AIC)

The proper libraries are `pmdarima.auto_arima` and `sktime` (both wrap a stepwise search).
They're offline here, so we reproduce the **identical logic** with a small grid search over
`statsmodels`, picking the order that **minimizes AIC**.

**Why AIC?** It scores the trade-off between **underfitting** (too simple → misses real
patterns) and **overfitting** (too complex → fits *noise*, fails on unseen data). AIC
**rewards good fit but penalizes complexity** — a scorecard where extra parameters cost points.

In [ ]:
def auto_arima_aic(series, max_p=2, max_d=2, max_q=2):
    series = series.dropna()
    best = (np.inf, None, None)
    for p in range(max_p+1):
        for d in range(max_d+1):
            for q in range(max_q+1):
                try:
                    aic = ARIMA(series, order=(p,d,q)).fit().aic
                    if aic < best[0]:
                        best = (aic, (p,d,q), None)
                except Exception:
                    continue
    return best[1], best[0]

# Select the best order on a 100-bar window for each asset (one-time selection)
sel = []
for name, ser in assets.items():
    order, aic = auto_arima_aic(ser.iloc[-100:] if len(ser) > 100 else ser)
    sel.append([name, order, round(aic,1), sum(order)])
selected = pd.DataFrame(sel, columns=["asset","auto_order(p,d,q)","AIC","complexity(p+d+q)"])
print(selected.to_string(index=False))

In [ ]:
# Did the AUTO order beat the FIXED (2,1,2)? Re-run the strategy with each asset's auto order.
rows = []
for name, ser in assets.items():
    order = selected.loc[selected.asset==name, "auto_order(p,d,q)"].iloc[0]
    sig = arima_signals(ser, order=order, n_test=40, window=60)
    st, bh, _ = evaluate(ser, sig)
    fx = fixed.loc[fixed.asset==name, "excess"].iloc[0]
    rows.append([name, str(order), round(st*100,1), round(bh*100,1),
                 round((st-bh)*100,1), round(fx*100,1),
                 "improved" if (st-bh) > fx else "worse/same"])
auto = pd.DataFrame(rows, columns=["asset","auto_order","strat%","b&h%","auto_excess%","fixed_excess%","vs fixed"])
print(auto.to_string(index=False))
print(f"\nAuto-selection improved on {(auto['vs fixed']=='improved').sum()} of {len(auto)} assets.")

**Honest verdict (matches the lecture):** automation often picks a *simpler* model
(e.g. `(0,1,0)` = a pure random walk) and **does not automatically beat the manual guess**.
AIC only balances under/over-fit on the **training data** — it never compares against buy &
hold or against profitability. Automation is a useful *tool*, **not** a silver bullet.

## 4. Transaction costs — where profits go to die

Every trade costs money: **commission + spread + slippage**. For a liquid US name the
instructor used ≈ **0.03% per trade** (`0.0003`). The cost depends on your **market** and
**broker**. Load it onto a profitable strategy and a chunk of the edge evaporates.

We use NIFTY weekly + a rolling `ARIMA(2,1,2)` one-step strategy, then subtract a cost every
time the **position changes** (a turn = entering/flipping).

In [ ]:
# Build a longer signal series for the cost study
series = px_w
sig = arima_signals(series, order=(2,1,2), n_test=120, window=80)
ret = series.pct_change().reindex(sig.index)
position = sig.shift(1).fillna(0)
gross = (position * ret).dropna()

COST = 0.0003                                  # 0.03% per trade
turns = position.diff().abs().fillna(0)        # 0 -> no change, 2 -> full flip (-1 to +1)
cost_series = (turns * COST).reindex(gross.index).fillna(0)
net = gross - cost_series

gross_tot = (1+gross).prod()-1
net_tot   = (1+net).prod()-1
bh_tot    = (1+ret.loc[gross.index]).prod()-1
n_trades  = int((turns > 0).sum())
total_cost = cost_series.sum()

print(f"Before costs : {gross_tot*100:6.1f}%")
print(f"After  costs : {net_tot*100:6.1f}%")
print(f"Buy & hold   : {bh_tot*100:6.1f}%")
print(f"Excess before: {(gross_tot-bh_tot)*100:6.1f}%  -> after: {(net_tot-bh_tot)*100:6.1f}%")
print(f"Number of trades: {n_trades}  | total cost drag: {total_cost*100:.2f}%")

The drag is small here because NIFTY (like the lecture's PayPal) is **very liquid** and we
trade infrequently. On a **less liquid** asset, or a high-turnover strategy, the cost line and
the gross line diverge sharply — and many "profitable" backtests turn into losers. **Always
backtest *net* of costs.**

## 5. Volatility is not constant — clustering & heteroscedasticity

So far risk was treated as **constant** — every signal went *all-in*. Reality: volatility
**changes over time and clusters**. Calm begets calm; storms beget storms (COVID 2020, rate
hikes 2022, wars). The technical name is **heteroscedasticity** ("different variance"):
unlike returns (nearly random day to day), **volatility has positive autocorrelation**.

We compute **rolling 30-day volatility**, annualized by × √252.

In [ ]:
rets_d = px_d.pct_change().dropna()
roll_vol = rets_d.rolling(30).std() * np.sqrt(252)      # annualized 30-day vol

# Regime table by period (our data is 2017-2020, COVID lands at the end)
regimes = {
    "2017 calm":        ("2017-01-01","2017-12-31"),
    "2018 wobble":      ("2018-01-01","2018-12-31"),
    "2019 pre-COVID":   ("2019-01-01","2019-12-31"),
    "2020 COVID":       ("2020-01-01","2020-12-31"),
}
rows = []
for name,(a,b) in regimes.items():
    seg = roll_vol.loc[a:b].dropna()
    if len(seg):
        rows.append([name, f"{seg.mean()*100:.0f}%", f"{seg.min()*100:.0f}%–{seg.max()*100:.0f}%"])
vol_table = pd.DataFrame(rows, columns=["period","avg annual vol","range"])
print(vol_table.to_string(index=False))
print(f"\nWhole-sample vol swings from {roll_vol.min()*100:.0f}% to {roll_vol.max()*100:.0f}% annualized.")

**Vol-based position sizing.** If you risk a *fixed* \$10,000 per trade, your real daily risk
is 4× larger in a high-vol regime than a low-vol one. Fix the **dollar risk** instead and let
**position size shrink when vol rises**.

In [ ]:
def daily_risk(position_size, annual_vol):
    return position_size * annual_vol / np.sqrt(252)

for label, av in [("low-vol  (20%)", 0.20), ("high-vol (100%)", 1.00)]:
    print(f"{label}: $10,000 position -> ${daily_risk(10000, av):6.0f} risked per day")

target_daily_risk = 250.0   # fix the dollar risk, solve for size
for label, av in [("low-vol  (20%)", 0.20), ("high-vol (100%)", 1.00)]:
    size = target_daily_risk * np.sqrt(252) / av
    print(f"{label}: to risk ${target_daily_risk:.0f}/day -> position size ${size:,.0f}")

## 6. Forecasting volatility — ARCH & GARCH

To size by volatility we must **predict** it. Two model families do exactly that:

- **ARCH** = **A**uto**R**egressive **C**onditional **H**etero**s**cedasticity (Engle, Nobel Prize).
  Today's variance depends on recent **shocks**:
  $$\sigma_t^2 = \omega + \alpha\,\epsilon_{t-1}^2$$
  i.e. *predicted variance = baseline ω + α × (yesterday's squared return)* — squared returns
  proxy for past variance.
- **GARCH** (Bollerslev) extends ARCH by also feeding back **past variance**:
  $$\sigma_t^2 = \omega + \alpha\,\epsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2$$
  This captures clustering with far fewer terms — the workhorse of volatility forecasting.

We fit a **GARCH(1,1)** on NIFTY daily returns with the `arch` library.

In [ ]:
r = rets_d * 100.0                          # arch likes percentage returns
garch = arch_model(r, mean="Constant", vol="GARCH", p=1, q=1, dist="t")
res = garch.fit(disp="off")
print(res.params.round(4).to_string())
omega, alpha, beta = res.params["omega"], res.params["alpha[1]"], res.params["beta[1]"]
print(f"\nalpha + beta = {alpha+beta:.3f}  (close to 1 => vol shocks are very persistent / cluster)")
cond_vol = res.conditional_volatility / 100 * np.sqrt(252)   # back to annualized fraction
cond_vol = pd.Series(cond_vol, index=r.index)

In [ ]:
# One-step-ahead variance forecast -> tomorrow's expected annualized vol
fc = res.forecast(horizon=1)
var_next = fc.variance.iloc[-1, 0]              # in (pct)^2
vol_next = np.sqrt(var_next) / 100 * np.sqrt(252)
print(f"GARCH forecast of NEXT-DAY annualized volatility: {vol_next*100:.1f}%")
print(f"Latest rolling 30d realized vol:                  {roll_vol.iloc[-1]*100:.1f}%")

**ARIMA + GARCH together.** ARIMA forecasts the **return/price**; GARCH forecasts the
**risk** around it. Combined, you get a *return signal sized by predicted volatility* — a
common and powerful pairing. (Further extensions — EGARCH, GJR-GARCH — are out of scope.)

The cricket-ticket analogy: a team on a winning run draws big crowds (high "demand variance")
that *cluster*; a losing run draws thin crowds. GARCH gauges which phase the market is in.

## 7. Summary chart — volatility clustering, GARCH fit, and the cost drag

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
G = "#16a34a"; DK = "#14532d"; RD = "#dc2626"; GY = "#9ca3af"

# (1) price with COVID shaded
ax[0,0].plot(px_d.index, px_d.values, color=G, lw=1.2)
ax[0,0].axvspan(pd.Timestamp("2020-02-15"), pd.Timestamp("2020-05-31"), color=RD, alpha=0.15)
ax[0,0].set_title("NIFTY price (COVID crash shaded)", fontweight="bold")
ax[0,0].text(pd.Timestamp("2020-03-01"), px_d.min(), " COVID", color=RD, fontsize=9)

# (2) rolling vol clustering
ax[0,1].plot(roll_vol.index, roll_vol.values*100, color=DK, lw=1.1)
ax[0,1].axhline(roll_vol.mean()*100, color=RD, ls="--", lw=1, label="mean vol")
ax[0,1].set_title("30-day annualized vol — clusters (heteroscedasticity)", fontweight="bold")
ax[0,1].set_ylabel("annual vol %"); ax[0,1].legend(fontsize=8)

# (3) GARCH conditional vol vs realized
ax[1,0].plot(roll_vol.index, roll_vol.values*100, color=GY, lw=1, label="realized 30d")
ax[1,0].plot(cond_vol.index, cond_vol.values*100, color=G, lw=1.2, label="GARCH(1,1)")
ax[1,0].set_title("GARCH conditional vol tracks the clustering", fontweight="bold")
ax[1,0].set_ylabel("annual vol %"); ax[1,0].legend(fontsize=8)

# (4) transaction-cost drag (equity gross vs net vs B&H)
eq_gross = (1+gross).cumprod(); eq_net = (1+net).cumprod()
eq_bh = (1+ret.loc[gross.index]).cumprod()
ax[1,1].plot(eq_gross.index, eq_gross.values, color=G, lw=1.3, label="strategy gross")
ax[1,1].plot(eq_net.index, eq_net.values, color=DK, lw=1.1, ls="--", label="strategy net of cost")
ax[1,1].plot(eq_bh.index, eq_bh.values, color=GY, lw=1.1, label="buy & hold")
ax[1,1].set_title("ARIMA(2,1,2) weekly: cost drag & vs B&H", fontweight="bold")
ax[1,1].legend(fontsize=8)

for a in ax.ravel(): a.grid(alpha=0.25)
plt.tight_layout(); plt.savefig("chart_1_arch.png", dpi=110, bbox_inches="tight")
print("saved chart_1_arch.png")

## 8. The one-paragraph version

Time-series modeling for trading adds four tools to ARIMA. **Automated order selection** (via
the AIC information criterion, balancing under- vs over-fit) replaces hand-picking `(p,d,q)` —
but it optimizes *fit*, not profit, and often **doesn't beat a manual guess**. **Transaction
costs** (~0.03%/trade) must be subtracted from every backtest; liquid, low-turnover strategies
survive, high-turnover ones often don't. **Volatility is not constant** — it **clusters**
(heteroscedasticity), so a fixed position size over- and under-risks by turns; fix the *dollar
risk* and size inversely to vol. To predict vol you need **ARCH** (variance from past shocks)
and **GARCH** (also from past variance), with `alpha+beta` near 1 confirming persistence.
**Combine ARIMA (forecasts return) with GARCH (forecasts risk)** for a volatility-sized signal.
The overriding lesson: **no silver bullets** — test everything, net of costs, on your own market.

# Additive validation layer

These cells are appended only in this validated copy.

In [ ]:
import pandas as pd
from pathlib import Path
base = Path('.')
files = ['asq02_html_audit.csv', 'asq02_source_inventory.csv', 'asq02_dependency_execution.csv', 'asq02_fixed_auto_arima_metrics.csv', 'asq02_cost_drag_metrics.csv', 'asq02_volatility_garch_metrics.csv', 'asq02_validation_controls.csv']
data = {f: pd.read_csv(base / f) for f in files}
{k: v.shape for k, v in data.items()}

## 1. Dependency and source audit

In [ ]:
print(data['asq02_source_inventory.csv'].to_string(index=False))
print(data['asq02_dependency_execution.csv'].to_string(index=False))
dep = data['asq02_dependency_execution.csv'].set_index('metric')
assert dep.loc['statsmodels_available','value'] in ['yes','no']

## 2. ARIMA-style and cost checks

In [ ]:
print(data['asq02_fixed_auto_arima_metrics.csv'].head(10).to_string(index=False))
print(data['asq02_cost_drag_metrics.csv'].to_string(index=False))
cost = data['asq02_cost_drag_metrics.csv'].set_index('metric')
assert float(cost.loc['net_total_pct','value']) < float(cost.loc['gross_total_pct','value'])

## 3. Volatility and controls

In [ ]:
print(data['asq02_volatility_garch_metrics.csv'].to_string(index=False))
print(data['asq02_validation_controls.csv'].to_string(index=False))
assert data['asq02_validation_controls.csv'].shape[0] >= 6